PULIZIA- togliamo i report duplicati. 
logica: Un report FAERS può apparire più volte nel database per tre motivi distinti, ognuno con logica di gestione diversa:
1. Follow-up report (il caso più comune)
Quando un evento avverso viene aggiornato — nuove informazioni, correzioni, esito finale — viene sottomesso un nuovo report con lo stesso safetyreportid ma receiptdate più recente. Nel database coesistono quindi la versione originale e una o più versioni aggiornate dello stesso caso.
Logica corretta: tieni solo la versione più recente per ogni safetyreportid.
2. Duplicati veri (stesso evento, reporter diversi)
Lo stesso evento viene riportato indipendentemente da più fonti — ad esempio sia dal medico che dall'azienda farmaceutica. Hanno safetyreportid diversi ma descrivono lo stesso paziente e stesso evento.
Logica corretta: questo è il problema hard. OpenVigil lo risolve con algoritmi probabilistici su combinazioni di campi (sesso + età + farmaco + reazione + data). Noi per ora lo approssimiamo con la strategia 1, che cattura la maggior parte dei duplicati.
3. Duplicati da overlap dei file bulk

Scaricando file di quarter diversi, un report con receivedate in Q1 potrebbe fisicamente trovarsi nel file Q2 se è stato processato in ritardo.
Logica corretta: stessa strategia del caso 1 — safetyreportid unico, versione più recente.

In [ ]:
"""
deduplicate_faers.py

Deduplicazione del dataset FAERS flat (faers_flat.parquet) per follow-up report
e overlap tra file bulk di quarter diversi.

Logica:
  - Per ogni safetyreportid non-null: tieni solo le righe con receivedate massima.
  - Per le righe con safetyreportid IS NULL (file legacy): passale invariate.

Output:
  - faers_flat_deduped.parquet  — dataset pulito
  - deduplication_report.txt    — report QC consultabile a fine progetto
"""

import duckdb
import platform
from pathlib import Path
from datetime import datetime

INPUT  = Path("data/faers_flat.parquet")
OUTPUT = Path("data/faers_flat_deduped.parquet")
REPORT = Path("data/deduplication_report.txt")

con = duckdb.connect()

run_ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# ── 1. DIAGNOSTICA PRE-DEDUPLICAZIONE ────────────────────────────────────────
print("=== PRE-DEDUPLICAZIONE ===")

pre = con.execute(f"""
    SELECT
        COUNT(*)                                                      AS total_rows,
        COUNT(DISTINCT safetyreportid)                               AS distinct_report_ids,
        COALESCE(SUM(CASE WHEN safetyreportid IS NULL THEN 1 END), 0) AS null_report_id_rows
    FROM '{INPUT}'
""").fetchone()
total_rows, distinct_ids, null_rows = pre

dup_reports = con.execute(f"""
    SELECT COUNT(*) FROM (
        SELECT safetyreportid
        FROM '{INPUT}'
        WHERE safetyreportid IS NOT NULL
        GROUP BY safetyreportid
        HAVING COUNT(DISTINCT receivedate) > 1
    )
""").fetchone()[0]

# Righe coinvolte nei report con duplicati (per calcolare il "risparmio")
rows_in_dup_reports = con.execute(f"""
    SELECT COUNT(*) FROM '{INPUT}'
    WHERE safetyreportid IN (
        SELECT safetyreportid
        FROM '{INPUT}'
        WHERE safetyreportid IS NOT NULL
        GROUP BY safetyreportid
        HAVING COUNT(DISTINCT receivedate) > 1
    )
""").fetchone()[0]

# Breakdown per quarter: quante righe appartengono a report che verranno deduplicati
quarter_breakdown_pre = con.execute(f"""
    SELECT receive_quarter,
           COUNT(*) AS total_rows,
           COUNT(DISTINCT safetyreportid) AS distinct_ids
    FROM '{INPUT}'
    WHERE safetyreportid IS NOT NULL
    GROUP BY receive_quarter
    ORDER BY receive_quarter
""").df()

print(f"  Righe totali:              {total_rows:>10,}")
print(f"  safetyreportid distinti:   {distinct_ids:>10,}")
print(f"  Righe con report_id NULL:  {null_rows:>10,}")
print(f"  Report con >1 receivedate: {dup_reports:>10,}")

# ── 2. DEDUPLICAZIONE ─────────────────────────────────────────────────────────
print("\n=== DEDUPLICAZIONE IN CORSO ===")

con.execute(f"""
COPY (
    WITH latest AS (
        SELECT safetyreportid, MAX(receivedate) AS max_date
        FROM '{INPUT}'
        WHERE safetyreportid IS NOT NULL
        GROUP BY safetyreportid
    )
    SELECT f.*
    FROM '{INPUT}' f
    JOIN latest l
      ON f.safetyreportid = l.safetyreportid
     AND f.receivedate    = l.max_date

    UNION ALL

    SELECT *
    FROM '{INPUT}'
    WHERE safetyreportid IS NULL
)
TO '{OUTPUT}' (FORMAT PARQUET)
""")

print(f"  Salvato: {OUTPUT}")

# ── 3. DIAGNOSTICA POST-DEDUPLICAZIONE ───────────────────────────────────────
print("\n=== POST-DEDUPLICAZIONE ===")

post = con.execute(f"""
    SELECT
        COUNT(*)                                                      AS total_rows,
        COUNT(DISTINCT safetyreportid)                               AS distinct_report_ids,
        COALESCE(SUM(CASE WHEN safetyreportid IS NULL THEN 1 END), 0) AS null_report_id_rows
    FROM '{OUTPUT}'
""").fetchone()
total_rows2, distinct_ids2, null_rows2 = post

residual_dups = con.execute(f"""
    SELECT COUNT(*) FROM (
        SELECT safetyreportid
        FROM '{OUTPUT}'
        WHERE safetyreportid IS NOT NULL
        GROUP BY safetyreportid
        HAVING COUNT(DISTINCT receivedate) > 1
    )
""").fetchone()[0]

quarter_breakdown_post = con.execute(f"""
    SELECT receive_quarter,
           COUNT(*) AS total_rows,
           COUNT(DISTINCT safetyreportid) AS distinct_ids
    FROM '{OUTPUT}'
    WHERE safetyreportid IS NOT NULL
    GROUP BY receive_quarter
    ORDER BY receive_quarter
""").df()

rows_removed   = total_rows - total_rows2
pct_removed    = 100 * rows_removed / total_rows if total_rows > 0 else 0

print(f"  Righe totali:              {total_rows2:>10,}")
print(f"  safetyreportid distinti:   {distinct_ids2:>10,}")
print(f"  Righe con report_id NULL:  {null_rows2:>10,}")
print(f"\n  Righe rimosse:             {rows_removed:>10,}  ({pct_removed:.1f}%)")
print(f"  Residual dups check:       {residual_dups}  ({'✓ OK' if residual_dups == 0 else '✗ ATTENZIONE'})")

# ── 4. REPORT TESTUALE ────────────────────────────────────────────────────────

# Prepara tabella quarter comparativa
merged = quarter_breakdown_pre.merge(
    quarter_breakdown_post,
    on="receive_quarter",
    suffixes=("_pre", "_post")
)
merged["rows_removed"] = merged["total_rows_pre"] - merged["total_rows_post"]
merged["pct_removed"]  = (100 * merged["rows_removed"] / merged["total_rows_pre"]).round(1)

col_q   = max(len("Quarter"), merged["receive_quarter"].str.len().max())
col_pre = max(len("Righe pre"), 10)
col_pos = max(len("Righe post"), 10)
col_rem = max(len("Rimosse"), 8)
col_pct = max(len("% rim."), 6)

def row_fmt(q, pre, post, rem, pct):
    return (f"  {str(q):<{col_q}}  {str(pre):>{col_pre}}  "
            f"{str(post):>{col_pos}}  {str(rem):>{col_rem}}  {str(pct):>{col_pct}}")

header = row_fmt("Quarter", "Righe pre", "Righe post", "Rimosse", "% rim.")
sep    = "  " + "-"*col_q + "  " + "-"*col_pre + "  " + "-"*col_pos + "  " + "-"*col_rem + "  " + "-"*col_pct

table_lines = [header, sep]
for _, r in merged.iterrows():
    table_lines.append(row_fmt(
        r["receive_quarter"],
        f"{int(r['total_rows_pre']):,}",
        f"{int(r['total_rows_post']):,}",
        f"{int(r['rows_removed']):,}",
        f"{r['pct_removed']}%"
    ))
table_lines.append(sep)
table_lines.append(row_fmt(
    "TOTALE",
    f"{total_rows:,}",
    f"{total_rows2:,}",
    f"{rows_removed:,}",
    f"{pct_removed:.1f}%"
))

report_text = f"""
================================================================================
FAERS DEDUPLICATION REPORT
================================================================================
Data esecuzione  : {run_ts}
Python / DuckDB  : {platform.python_version()} / {duckdb.__version__}
File input       : {INPUT}
File output      : {OUTPUT}

--------------------------------------------------------------------------------
1. CONTESTO E MOTIVAZIONE
--------------------------------------------------------------------------------
Il database FAERS (FDA Adverse Event Reporting System) contiene report di eventi
avversi che possono apparire più volte per tre ragioni distinte:

  a) Follow-up report: lo stesso caso viene aggiornato nel tempo con nuove
     informazioni (es. esito finale, correzioni). Ogni aggiornamento genera un
     nuovo record con lo stesso safetyreportid ma receivedate più recente.

  b) Duplicati veri: lo stesso evento viene riportato indipendentemente da più
     fonti (medico, azienda farmaceutica). Hanno safetyreportid diversi ma
     descrivono lo stesso paziente/evento. Richiedono algoritmi probabilistici
     per essere identificati (fuori scope di questa pipeline).

  c) Overlap tra file bulk: scaricando file di quarter diversi, un report con
     receivedate in Q1 può trovarsi fisicamente nel file Q2 se processato in
     ritardo dalla FDA.

Questa pipeline gestisce i casi (a) e (c) con la stessa logica: per ogni
safetyreportid, mantieni solo le righe corrispondenti alla receivedate massima.

--------------------------------------------------------------------------------
2. LOGICA DI DEDUPLICAZIONE
--------------------------------------------------------------------------------
Schema del parquet: ogni riga rappresenta una coppia (drug, reaction) all'interno
di un report. Un singolo safetyreportid genera N righe (una per ogni combinazione
farmaco-reazione). La deduplicazione opera a livello di REPORT, non di riga:
per ogni safetyreportid si identifica la receivedate massima e si eliminano
tutte le righe con date precedenti.

  Query DuckDB equivalente:
    WITH latest AS (
        SELECT safetyreportid, MAX(receivedate) AS max_date
        FROM faers_flat
        WHERE safetyreportid IS NOT NULL
        GROUP BY safetyreportid
    )
    SELECT f.*
    FROM faers_flat f
    JOIN latest l
      ON f.safetyreportid = l.safetyreportid
     AND f.receivedate    = l.max_date

  Casi non deduplicati (per scelta):
    - Righe con safetyreportid IS NULL (file FAERS legacy pre-2012): non
      deduplicabili per ID. Mantenute invariate. Rappresentano il {100*null_rows/total_rows:.1f}%
      delle righe totali. Questa limitazione è documentata e accettata.
    - Report con stesso safetyreportid E stessa receivedate ma dati divergenti
      (duplicati veri): entrambe le righe sono mantenute, in assenza di algoritmi
      probabilistici per disambiguarle.

--------------------------------------------------------------------------------
3. RISULTATI
--------------------------------------------------------------------------------

  Righe totali (pre)          : {total_rows:>12,}
  safetyreportid distinti     : {distinct_ids:>12,}
  Report con >1 receivedate   : {dup_reports:>12,}   (questi generano righe duplicate)
  Righe in report duplicati   : {rows_in_dup_reports:>12,}

  Righe totali (post)         : {total_rows2:>12,}
  Righe rimosse               : {rows_removed:>12,}  ({pct_removed:.1f}% del totale)
  Righe NULL report_id        : {null_rows2:>12,}   (invariate, come atteso)

  Verifica residui (>1 receivedate per report_id): {residual_dups} {'✓ PASS' if residual_dups == 0 else '✗ FAIL — investigare'}

  Breakdown per quarter:

{chr(10).join(table_lines)}

--------------------------------------------------------------------------------
4. LIMITAZIONI DOCUMENTATE
--------------------------------------------------------------------------------
  - I duplicati veri (caso b) NON sono gestiti da questa pipeline. La loro
    rimozione richiederebbe algoritmi probabilistici (simili a quelli di
    OpenVigil) basati su combinazioni di: sesso, età, farmaco, reazione, data.
    L'impatto stimato in letteratura è <5% dei report totali.

  - I report con safetyreportid NULL non possono essere deduplicati e sono
    mantenuti as-is. Rappresentano principalmente dati storici (pre-2012) e
    costituiscono il {100*null_rows/total_rows:.1f}% del dataset.

  - Nei rari casi in cui lo stesso safetyreportid ha più righe con la stessa
    receivedate massima ma contenuto divergente, tutte le righe vengono
    mantenute (comportamento conservativo).

--------------------------------------------------------------------------------
5. FILE PRODOTTI
--------------------------------------------------------------------------------
  {OUTPUT}
    Parquet deduplicato, pronto per le analisi di disproportionality
    (PRR, ROR, BCPNN). Schema identico a faers_flat.parquet.

================================================================================
END OF REPORT
================================================================================
""".strip()

REPORT.write_text(report_text, encoding="utf-8")
print(f"\n  Report salvato: {REPORT}")

con.close()